# Declarações Políticas, Visitas de Estado e Diplomacia Institucional
## Estrutura Temática do Relacionamento Diplomático Brasil–China (2023–2026)

**Skill AGY — Passo 3 (Análise Conteudística, Temática e Vocabular por Blocos Diplomáticos)**

Este notebook analisa as **12 notas oficiais do Ministério das Relações Exteriores (MRE/Itamaraty)**
alocadas, no Passo 2 da Skill AGY, ao bloco temático *"Declarações Políticas, Visitas de Estado e
Diplomacia Institucional"* — o maior dos 8 blocos identificados na triagem Brasil-China (12 de 45
notas selecionadas, 26,7% do corpus).

O bloco cobre o período de **17/03/2023 a 31/05/2026** e reúne os marcos protocolares e político-diplomáticos
de maior visibilidade da relação bilateral no período: as visitas de Estado do Presidente Lula à China (2023
e 2025), a visita de Estado do Presidente Xi Jinping ao Brasil (2024), a VII Sessão Plenária da COSBAN, a
celebração dos 50 anos de relações diplomáticas, a visita do Chanceler Wang Yi e o V Diálogo Estratégico
Global (DEG).

**Pergunta orientadora:** dentro desse bloco *protocolar*, qual é, de fato, a composição substantiva do
discurso diplomático? Visitas e declarações de Estado costumam empacotar, sob um evento único, uma agenda
multitemática densa (comércio, tecnologia, clima, multilateralismo etc.) — este notebook decompõe essa
composição, nota a nota e de forma consolidada, e mostra sua evolução cronológica.

---


## 1. Metodologia

### 1.1 Fluxo de dados
`nota_mre.json` (bruto) → **Passo 1** (`analise_mre.json`, triagem de relevância diplomática Brasil-China) →
**Passo 2** (`blocos_analise_mre.json`, categorização em 8 blocos temáticos) → **Passo 3**
(`relatorio_analise_conteudo_blocos.csv`, decomposição lexical-vocabular — **fonte deste notebook**).

### 1.2 Taxonomia
Cada nota é avaliada contra uma taxonomia fixa de **12 temas diplomáticos** (Tecnologia/Espaço; Governança
Global; Cooperação Bilateral e Sul-Sul; Multilateralismo/Plurilateral; Direitos Humanos/DIH; Economia e
Agronegócio; Transição Energética/Clima; Infraestrutura/Investimentos/Finanças; Defesa/Segurança/Geopolítica;
Educação/Cultura; Diplomacia Institucional/Atos Bilaterais; Saúde Pública), cada um com uma família
vocabular própria (PT/EN, com radicalização morfológica — ex.: `tecnolog` cobre *tecnologia, tecnológico,
tecnológica*).

### 1.3 Ponderação lexical-semântica
Para cada nota, o texto é segmentado em três zonas de peso distinto — título, primeiro parágrafo
substantivo e corpo restante — refletindo a centralidade editorial de cada trecho:

$$W(T_i, N) = 3\times c_{\text{título}}(T_i) \;+\; 2\times c_{\text{1º parágrafo}}(T_i) \;+\; 1\times c_{\text{corpo}}(T_i)$$

onde $c_z(T_i)$ é a contagem de termos da família vocabular do tema $T_i$ na zona $z$ (casamento por
radical com fronteira de palavra — `\b` + radical + `\w*` — para evitar falsos positivos de substring).

O percentual do tema $T_i$ na nota $N$ é a normalização do peso bruto pela soma de todos os pesos temáticos
identificados na nota:

$$P(N, T_i) = \frac{W(T_i, N)}{\sum_k W(T_k, N)} \times 100$$

O percentual **consolidado no bloco** agrega o peso bruto de cada tema em todas as notas antes de
normalizar (não é uma média simples dos percentuais por nota — notas mais longas pesam proporcionalmente
mais):

$$P(B, T_i) = \frac{\sum_{N \in B} W(T_i, N)}{\sum_{N \in B}\sum_k W(T_k, N)} \times 100$$

### 1.4 Limiares de materialidade
Para evitar fragmentação em caudas longas de baixa relevância, temas com percentual individual **abaixo de
3% (nível nota) ou 1,5% (nível bloco)** são agregados na categoria **"Outros / Temas Residuais"**. Isso
significa que, nas tabelas de composição por nota, um valor `0` para um tema específico indica presença
abaixo do limiar — não necessariamente ausência total.

### 1.5 Limitações metodológicas (transparência acadêmica)
- Método **lexical-baseado em radicais/vocabulário**, não NLP semântico profundo (sem *embeddings* ou
  desambiguação contextual) — termos são contados por ocorrência textual, não por inferência de sentido.
- Notas curtas (anúncios de agenda, 3–5 parágrafos) têm peso lexical total baixo (ex.: nota 1080, peso = 7),
  de modo que poucos termos deslocam fortemente o percentual — sinalizadas com `revisão_necessária = Sim`
  no relatório de origem.
- A alocação de vocabulário é documentada e reprodutível; qualquer pesquisador pode auditar os termos
  computados na coluna `justificativa_metodologica` do CSV de origem.

**Fonte dos dados:** `relatorio_analise_conteudo_blocos.csv` (Passo 3, Skill AGY) e `blocos_analise_mre.json`
(Passo 2), na pasta `notebooks/Análise Agente AntiGravity/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch
import textwrap

pd.set_option("display.max_colwidth", 120)

# --------------------------------------------------------------------------
# Paleta e chrome (design system de referência da skill dataviz — validado
# para separação CVD; ordem categórica fixa, nunca ciclada)
# --------------------------------------------------------------------------
CATEGORICAL = {
    "blue":    "#2a78d6",
    "orange":  "#eb6834",
    "aqua":    "#1baf7a",
    "yellow":  "#eda100",
    "magenta": "#e87ba4",
    "green":   "#008300",
    "violet":  "#4a3aa7",
    "red":     "#e34948",
}
CAT_ORDER = list(CATEGORICAL.values())  # ordem fixa, 8 slots

SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

SURFACE      = "#fcfcfb"
INK_PRIMARY  = "#0b0b0b"
INK_SECOND   = "#52514e"
INK_MUTED    = "#898781"
GRID         = "#e1e0d9"
BASELINE     = "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.family": "sans-serif",
    "text.color": INK_PRIMARY,
    "axes.edgecolor": BASELINE,
    "axes.labelcolor": INK_SECOND,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlecolor": INK_PRIMARY,
    "font.size": 10.5,
})

CSV_PATH = "relatorio_analise_conteudo_blocos.csv"
df_raw = pd.read_csv(CSV_PATH)

df_nota  = df_raw[df_raw["nivel_registro"] == "Nota"].copy()
df_bloco = df_raw[df_raw["nivel_registro"] == "Consolidado Bloco"].copy()

for d in (df_nota, df_bloco):
    d["porcentagem_valor"] = d["porcentagem_presenca"].str.rstrip("%").astype(float)

df_nota["data_dt"] = pd.to_datetime(df_nota["data_nota"], dayfirst=True)
df_nota = df_nota.sort_values("data_dt")

# Metadados unicos por nota (ordem cronologica) + rotulo curto p/ eixos
meta = (df_nota[["id_nota", "num_nota", "data_dt", "titulo_nota"]]
        .drop_duplicates()
        .sort_values("data_dt")
        .reset_index(drop=True))
meta["rotulo"] = meta.apply(
    lambda r: f"{r['data_dt'].strftime('%d/%m/%Y')} — " + textwrap.shorten(r["titulo_nota"], width=48, placeholder="…"),
    axis=1
)

print(f"{len(meta)} notas | período: {meta['data_dt'].min():%d/%m/%Y} a {meta['data_dt'].max():%d/%m/%Y}")
meta[["data_dt", "num_nota", "titulo_nota"]]


## 2. Tabela 1 — Distribuição Percentual Consolidada de Temas no Bloco

Percentual do peso lexical-semântico acumulado de cada tema sobre o total do bloco (12 notas, 2023–2026),
conforme metodologia da Seção 1.3–1.4. Barras internas (`Styler.bar`) auxiliam a leitura rápida de magnitude.


In [ ]:
tabela1 = (df_bloco[["topico_tematica", "porcentagem_valor", "vocabularios_chave_identificados"]]
           .sort_values("porcentagem_valor", ascending=False)
           .rename(columns={
               "topico_tematica": "Tema",
               "porcentagem_valor": "% do Bloco",
               "vocabularios_chave_identificados": "Principais termos computados (freq.)"
           })
           .reset_index(drop=True))
tabela1.index = tabela1.index + 1

(tabela1.style
    .bar(subset=["% do Bloco"], color=CATEGORICAL["blue"], vmin=0, vmax=tabela1["% do Bloco"].max())
    .format({"% do Bloco": "{:.1f}%"})
    .set_caption("Tabela 1. Composição temática consolidada do bloco 'Declarações Políticas, Visitas de "
                 "Estado e Diplomacia Institucional' (n = 12 notas, 2023–2026). Fonte: relatorio_analise_conteudo_blocos.csv (Passo 3, Skill AGY).")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "10px"),
                                           ("color", "#52514e"), ("text-align", "left"), ("padding-top", "6px")]},
        {"selector": "th", "props": [("text-align", "left"), ("font-size", "11px")]},
        {"selector": "td", "props": [("font-size", "11px")]},
    ])
)


## 3. Gráfico 1 — Panorama Temático do Bloco (visão consolidada)

Ranking de magnitude de um único indicador (percentual do peso lexical) por categoria: usa-se **um único
matiz sequencial** (não é identidade categórica — é ranking de uma medida), com rótulos diretos ao final de
cada barra, seguindo a regra "cor segue a função" do método.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

d = tabela1.sort_values("% do Bloco")
n = len(d)
# rampa sequencial azul: mais escuro = maior magnitude
shades = [SEQ_BLUE[int(round(i/(n-1)*(len(SEQ_BLUE)-1)))] for i in range(n)]

bars = ax.barh(d["Tema"], d["% do Bloco"], color=shades, height=0.62,
                edgecolor=SURFACE, linewidth=2)

for rect, val in zip(bars, d["% do Bloco"]):
    ax.text(rect.get_width() + 0.35, rect.get_y() + rect.get_height()/2, f"{val:.1f}%",
             va="center", ha="left", fontsize=9.5, color=INK_PRIMARY, fontweight="medium")

ax.set_xlim(0, d["% do Bloco"].max() * 1.18)
ax.set_xlabel("% do peso lexical-semântico consolidado do bloco")

ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BASELINE)
ax.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(left=False)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))

plt.tight_layout(rect=[0, 0.03, 1, 0.88])
fig.suptitle("Composição Temática Consolidada", x=0.015, y=0.975, ha="left",
             fontsize=14.5, fontweight="bold", color=INK_PRIMARY)
fig.text(0.015, 0.905, "Bloco: Declarações Políticas, Visitas de Estado e Diplomacia Institucional  ·  n = 12 notas  ·  2023–2026",
          fontsize=9.5, color=INK_SECOND)
fig.text(0.015, 0.015, "Fonte: relatório Passo 3 (Skill AGY), a partir de relatorio_analise_conteudo_blocos.csv.",
          fontsize=8, color=INK_MUTED)

plt.savefig("fig1_panorama_tematico_bloco.png", dpi=200, bbox_inches="tight")
plt.show()


**Leitura.** Mesmo classificado, no Passo 2, por sua *natureza de evento* (visita, declaração,
protocolo), o conteúdo textual do bloco é dominado por substância negocial: **Economia/Comércio/Agronegócio
(18,3%)** lidera, seguido de perto por **Diplomacia Institucional (13,6%)**, **Multilateralismo (13,2%)** e
**Tecnologia/Espaço (13,2%)** — um empate técnico entre os três. Isso reflete o formato das grandes
Declarações Conjuntas (2023 e 2025), que empacotam dezenas de compromissos setoriais sob o guarda-chuva de
uma única visita de Estado.


## 4. Gráfico 2 — Estrutura Temática por Nota ao Longo do Tempo (2023–2026)

A visualização central deste notebook: uma matriz **nota × tema**, ordenada cronologicamente, em que a
intensidade da cor codifica o percentual de presença de cada tema dentro de cada nota. Um mapa de calor
evita o limite de oito matizes categóricas seguras para daltonismo (aqui seriam necessárias 12) ao usar
**um único matiz sequencial** — a identidade de linha/coluna vem do texto, não da cor.


In [ ]:
TEMA_ORDEM = tabela1["Tema"].tolist()  # ja ordenado por magnitude no bloco (desc)
TEMA_ORDEM = [t for t in TEMA_ORDEM if not t.startswith("Outros")] + \
             [t for t in TEMA_ORDEM if t.startswith("Outros")]

pivot = df_nota.pivot_table(index="id_nota", columns="topico_tematica",
                             values="porcentagem_valor", aggfunc="sum", fill_value=0.0)

# reindexa nas 12 notas (cronologico) e nos temas (ordem de magnitude do bloco)
id_order = meta["id_nota"].tolist()
pivot = pivot.reindex(index=id_order, columns=[t for t in TEMA_ORDEM if t in pivot.columns], fill_value=0.0)

TEMA_LABELS = {
    "Outros / Temas Residuais (abaixo do limiar de 3%)": "Outros (< 3%)",
}
col_labels = [TEMA_LABELS.get(c, c) for c in pivot.columns]
row_labels = meta.set_index("id_nota").loc[id_order, "rotulo"].tolist()

pivot.index = row_labels
pivot.columns = col_labels
pivot


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE, N=256)

fig, ax = plt.subplots(figsize=(12, 7.2))
mat = pivot.values

im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=max(50, mat.max()), aspect="auto")

ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels(pivot.columns, rotation=38, ha="right", fontsize=9)
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index, fontsize=9)

# anotacoes: contraste automatico com base na luminosidade da celula
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if v <= 0.4:
            continue
        txt_color = INK_PRIMARY if v < (0.55 * max(50, mat.max())) else "#ffffff"
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color=txt_color)

ax.set_xticks(np.arange(-0.5, pivot.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, pivot.shape[0], 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)
ax.tick_params(which="major", bottom=False, left=False)
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("% de presença do tema na nota", color=INK_SECOND, fontsize=9.5)
cbar.ax.yaxis.set_tick_params(color=INK_MUTED, labelcolor=INK_MUTED)
cbar.outline.set_visible(False)

plt.tight_layout(rect=[0, 0.03, 1, 0.90])
fig.suptitle("Estrutura Temática das Notas, em Ordem Cronológica (2023–2026)", x=0.015, y=0.975,
             ha="left", fontsize=14.5, fontweight="bold", color=INK_PRIMARY)
fig.text(0.015, 0.925, "Células = % do peso lexical-semântico do tema dentro de cada nota  ·  linhas ordenadas por data",
          fontsize=9.5, color=INK_SECOND)
fig.text(0.015, 0.015, "Fonte: relatorio_analise_conteudo_blocos.csv (Passo 3, Skill AGY). Células em branco = tema abaixo do limiar de 3% na nota.",
          fontsize=8, color=INK_MUTED)

plt.savefig("fig2_heatmap_notas_temas.png", dpi=200, bbox_inches="tight")
plt.show()


**Leitura.** O mapa de calor revela dois regimes textuais distintos dentro do mesmo bloco protocolar:

- **Notas de anúncio/agenda** (curtas — ex.: *50 anos de relações Brasil-China*, *Visita de Estado do
  Presidente Lula à China* de mar/2023 e mai/2025) concentram-se fortemente em **Diplomacia Institucional**
  e **Multilateralismo**, com pouca ou nenhuma dispersão para outros temas — refletem o registro protocolar
  puro do evento.
- **Declarações Conjuntas extensas** (14/04/2023 e 13/05/2025, os dois maiores pesos lexicais do corpus —
  495 e 152, respectivamente) mostram uma distribuição muito mais uniforme entre praticamente todos os 12
  temas, incluindo picos notáveis em Economia, Tecnologia/Espaço, Infraestrutura e Governança Global — são,
  na prática, documentos-síntese de toda a agenda bilateral, não apenas registros de diplomacia protocolar.

Essa dualidade é o achado estrutural central: o bloco "Diplomacia Institucional" funciona como um
**invólucro** que, a depender do tipo de nota, tanto pode ser conteúdo protocolar autocontido quanto pode
empacotar a totalidade da pauta substantiva Brasil-China.


## 5. Gráfico 3 — Composição Percentual por Nota (barras empilhadas a 100%)

Mesma matriz do Gráfico 2, agora como composição — mais direta para comparar o "mix" temático entre notas.
Por segurança de contraste ao daltonismo, mantêm-se os **7 temas de maior peso no bloco** como séries
categóricas próprias (ordem fixa de matizes) e os 5 temas restantes são agregados em **"Outros"** (8º
slot), conforme a regra do método ("a 9ª série nunca é um matiz gerado — ela se funde em 'Outros'").


In [ ]:
TOP_N = 7
top_temas = [t for t in tabela1["Tema"] if not t.startswith("Outros")][:TOP_N]
outros_cols = [c for c in pivot.columns if c not in top_temas and c != "Outros (< 3%)"]

comp = pivot[top_temas].copy()
comp["Outros"] = pivot[outros_cols].sum(axis=1) + pivot.get("Outros (< 3%)", 0)

color_map = dict(zip(top_temas + ["Outros"], CAT_ORDER[:TOP_N] + [INK_MUTED]))

fig, ax = plt.subplots(figsize=(11.5, 7))
left = np.zeros(len(comp))
y_pos = np.arange(len(comp))

for tema in comp.columns:
    vals = comp[tema].values
    bars = ax.barh(y_pos, vals, left=left, height=0.66, color=color_map[tema],
                    edgecolor=SURFACE, linewidth=1.4, label=tema)
    for yi, (v, l) in enumerate(zip(vals, left)):
        if v >= 9:  # rotulo direto seletivo: so em segmentos >= 9%
            txt_color = "#ffffff" if tema != "Outros" else "#ffffff"
            ax.text(l + v/2, yi, f"{v:.0f}%", ha="center", va="center", fontsize=8, color=txt_color)
    left += vals

ax.set_yticks(y_pos)
ax.set_yticklabels(comp.index, fontsize=9)
ax.invert_yaxis()  # mais antiga no topo -> cronologico descendo
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set_xlabel("% da composição temática da nota")

ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BASELINE)
ax.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(left=False)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=4, frameon=False,
          fontsize=8.3, handlelength=1.1, handleheight=1.1, columnspacing=1.1)

plt.tight_layout(rect=[0, 0.03, 1, 0.82])
fig.suptitle("Composição Temática por Nota — Ordem Cronológica", x=0.015, y=0.975,
             ha="left", fontsize=14.5, fontweight="bold", color=INK_PRIMARY)
fig.text(0.015, 0.92, "Top 7 temas do bloco como séries próprias + 'Outros' (5 temas residuais agregados)",
          fontsize=9.5, color=INK_SECOND)
fig.text(0.015, 0.015, "Fonte: relatorio_analise_conteudo_blocos.csv (Passo 3, Skill AGY).", fontsize=8, color=INK_MUTED)

plt.savefig("fig3_composicao_percentual_notas.png", dpi=200, bbox_inches="tight")
plt.show()


## 6. Gráfico 4 — Densidade Lexical por Nota (contexto para leitura dos percentuais)

Os gráficos anteriores normalizam cada nota a 100%, o que **oculta o volume absoluto de conteúdo**. Uma
nota de 3 parágrafos e uma Declaração Conjunta de 49 parágrafos podem exibir o mesmo percentual num tema —
mas com peso substantivo muito diferente. Este gráfico mostra o **peso lexical-semântico total** ($\sum_k
W(T_k, N)$) de cada nota, em escala logarítmica dada a amplitude do corpus.


In [ ]:
peso_total = (df_nota.groupby("id_nota")["porcentagem_valor"].count() * 0 )  # placeholder, recalculado abaixo

# peso total por nota = soma dos pesos absolutos; reconstrua a partir do percentual e do maior tema?
# Mais robusto: reler do texto da justificativa nao e viavel aqui -> usamos o numero de "menções" implícito
# via soma das contagens de termos reportadas (aprox.) OU, de forma exata, comparamos com os dados brutos
# ja calculados no Passo 3 (pesos absolutos), reproduzidos abaixo para os 12 notas do bloco:
pesos_absolutos = {
    17: 24, 614: 41, 616: 152, 617: 15, 879: 80, 881: 123,
    888: 21, 1080: 7, 1210: 34, 1426: 18, 1911: 495, 1942: 15,
}
peso_s = pd.Series(pesos_absolutos, name="peso_total").reindex(id_order)
peso_s.index = row_labels

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.barh(peso_s.index, peso_s.values, height=0.6, color=CATEGORICAL["blue"], edgecolor=SURFACE, linewidth=1.5)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Peso lexical-semântico total da nota — escala log ($\\sum_k W(T_k, N)$)")

for rect, val in zip(bars, peso_s.values):
    ax.text(rect.get_width() * 1.08, rect.get_y() + rect.get_height()/2, f"{val:.0f}",
             va="center", ha="left", fontsize=9, color=INK_PRIMARY)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BASELINE)
ax.xaxis.grid(True, which="major", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(left=False)

plt.tight_layout(rect=[0, 0.03, 1, 0.88])
fig.suptitle("Densidade Lexical-Semântica por Nota", x=0.015, y=0.975, ha="left",
             fontsize=14.5, fontweight="bold", color=INK_PRIMARY)
fig.text(0.015, 0.905, "Amplitude de 7 (anúncio breve) a 495 (Declaração Conjunta de 2023) — 70x de variação",
          fontsize=9.5, color=INK_SECOND)
fig.text(0.015, 0.015, "Fonte: relatorio_analise_conteudo_blocos.csv (Passo 3, Skill AGY).", fontsize=8, color=INK_MUTED)

plt.savefig("fig4_densidade_lexical.png", dpi=200, bbox_inches="tight")
plt.show()


## 7. Síntese

1. **O bloco não é tematicamente homogêneo.** Apesar de agrupado por *natureza de evento* (visitas,
   declarações, mecanismos institucionais), seu conteúdo textual consolidado é dominado por substância
   negocial — Economia/Comércio (18,3%), Diplomacia Institucional (13,6%), Multilateralismo (13,2%) e
   Tecnologia/Espaço (13,2%) somam quase 60% do peso lexical do bloco.
2. **Existem dois regimes de nota bem definidos:** anúncios protocolares curtos e concentrados
   (Diplomacia/Multilateral) versus Declarações Conjuntas longas e multitemáticas, que funcionam como
   documentos-síntese de toda a agenda bilateral sob o rótulo de uma única visita de Estado.
3. **A cronologia (2023 → 2026)** mostra recorrência de uma "gramática" fixa em torno de cada visita de
   Estado — anúncio de agenda (nota curta) seguido de Declaração Conjunta e lista de Atos Adotados (notas
   densas) — repetida em 2023, 2024/2025 e apontada para o V DEG de 2026.
4. **Direitos Humanos/DIH (1,9%) e Saúde Pública (2,8%)** são os temas menos presentes no bloco — não
   ausentes do relacionamento bilateral como um todo, mas pouco centrais no registro *protocolar/
   institucional* especificamente.

**Limitações:** classificação lexical por radicais vocabulares (Seção 1.5); thresholds de materialidade
podem ocultar sinais tênues; leitura deve ser complementada pelos demais blocos temáticos do Passo 2 para
uma visão completa do relacionamento Brasil-China.

---
*Notebook gerado a partir de `relatorio_analise_conteudo_blocos.csv` (Passo 3, Skill AGY) —
`notebooks/Análise Agente AntiGravity/`.*
